Assignment 3 \
PySpark DataFrame Case Study: Employee Performance Review Analysis \
Scenario: You have a dataset containing employee performance reviews over time and want to analyze performance trends, average ratings, and identify high performers. The dataset contains the following columns: \
• emp_id: Employee ID \
• review_date: Date of the performance review \
• department: Department name \
• rating: Performance rating (1-5 scale) \
• reviewer: Name of the person conducting the review \
Sample Data


| emp_id | review_date | department  | rating|  reviewer  |\
|   1    | 2024-01-10  | Engineering |   5   |   John     |\
|   2    | 2024-01-11  | HR          |   4   |   Jane     |\
|   3    | 2024-01-12  | Sales       |   3   |   Sam      |\
|   4    | 2024-02-01  | Engineering |   5   |   John     |\
|   1    | 2024-03-10  | Engineering |   4   |   Jane     |\
|   2    | 2024-03-11  | HR          |  NULL |   Sam      |



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, coalesce
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.appName("Emp").getOrCreate()

In [0]:
data = [
    (1, '2024-01-10', 'Engineering', 5, 'John'),
    (2, '2024-01-11', 'HR', 4, 'Jane'),
    (3, '2024-01-12', 'Sales', 3, 'Sam'),
    (4, '2024-02-01', 'Engineering', 5, 'John'),
    (1, '2024-03-10', 'Engineering', 4, 'Jane'),
    (2, '2024-03-11', 'HR', None, 'Sam')
]

# Create DataFrame
columns = ["emp_id", "review_date", "department", "rating", "reviewer"]
df = spark.createDataFrame(data, schema=columns)
df.show()


+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
# Find all reviews where the ratings was 4 or higher

df_filtered = df.filter(df.rating >= 4)
df_filtered.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
+------+-----------+-----------+------+--------+



In [0]:
# Fill null values in the 'rating' column with the department average rating.

window_spec = Window.partitionBy('department')
df_filled = df.withColumn('rating', coalesce(df.rating, avg('rating').over(window_spec)))
df_filled.show()


+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|   5.0|    John|
|     4| 2024-02-01|Engineering|   5.0|    John|
|     1| 2024-03-10|Engineering|   4.0|    Jane|
|     2| 2024-01-11|         HR|   4.0|    Jane|
|     2| 2024-03-11|         HR|   4.0|     Sam|
|     3| 2024-01-12|      Sales|   3.0|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
# Remove duplicate reviews by 'emp_id' and 'review_date'.

df_no_duplicates = df.dropDuplicates(['emp_id', 'review_date'])
df_no_duplicates.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
# Select 'emp_id', 'department', and 'rating' columns.

df_selected = df.select('emp_id', 'department', 'rating')
df_selected.show()


+------+-----------+------+
|emp_id| department|rating|
+------+-----------+------+
|     1|Engineering|     5|
|     2|         HR|     4|
|     3|      Sales|     3|
|     4|Engineering|     5|
|     1|Engineering|     4|
|     2|         HR|  NULL|
+------+-----------+------+



In [0]:
# Calculate the average rating per department.

df_grouped = df.groupBy('department').agg({'rating': 'avg'})
df_grouped.show()


+-----------+-----------------+
| department|      avg(rating)|
+-----------+-----------------+
|Engineering|4.666666666666667|
|         HR|              4.0|
|      Sales|              3.0|
+-----------+-----------------+



In [0]:
#Join with another DataFrame 'df_employees' containing 'emp_id' and 'employee_name'.
# Employee details dataframe
data2 = [
    (1, "John"),
    (2, "Alice"),
    (3, "Bob")
]

columns2 = ["emp_id", "employee_name"]

df_employees = spark.createDataFrame(data2, columns2)

# Assuming df_employees is another DataFrame that contains employee details
df_joined = df.join(df_employees, on='emp_id', how='inner')
df_joined.show()


+------+-----------+-----------+------+--------+-------------+
|emp_id|review_date| department|rating|reviewer|employee_name|
+------+-----------+-----------+------+--------+-------------+
|     1| 2024-01-10|Engineering|     5|    John|         John|
|     2| 2024-01-11|         HR|     4|    Jane|        Alice|
|     3| 2024-01-12|      Sales|     3|     Sam|          Bob|
|     1| 2024-03-10|Engineering|     4|    Jane|         John|
|     2| 2024-03-11|         HR|  NULL|     Sam|        Alice|
+------+-----------+-----------+------+--------+-------------+



In [0]:
#Union with another 'df_new_reviews' DataFrame containing additional reviews.
# Additional reviews data
new_data = [
    (5, '2024-04-01', 'Finance', 5, 'Alice'),
    (6, '2024-04-02', 'Marketing', 4, 'Bob')
]

# Create new DataFrame
df_new_reviews = spark.createDataFrame(new_data, schema=columns)

# Union both DataFrames
df_union = df.union(df_new_reviews)

# Show combined result
df_union.show()


+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
|     5| 2024-04-01|    Finance|     5|   Alice|
|     6| 2024-04-02|  Marketing|     4|     Bob|
+------+-----------+-----------+------+--------+



In [0]:
#Create a temp view and find the average rating for each employee.

df.createOrReplaceTempView('performance_reviews')
sql_result = spark.sql('SELECT emp_id, AVG(rating) as avg_rating FROM performance_reviews GROUP BY emp_id')
sql_result.show()


+------+----------+
|emp_id|avg_rating|
+------+----------+
|     1|       4.5|
|     2|       4.0|
|     3|       3.0|
|     4|       5.0|
+------+----------+



In [0]:
#Calculate the cumulative average rating for each employee over time.

window_spec = Window.partitionBy('emp_id').orderBy('review_date')
df_with_cumulative_avg = df.withColumn('cumulative_avg', avg('rating').over(window_spec))
df_with_cumulative_avg.show()


+------+-----------+-----------+------+--------+--------------+
|emp_id|review_date| department|rating|reviewer|cumulative_avg|
+------+-----------+-----------+------+--------+--------------+
|     1| 2024-01-10|Engineering|     5|    John|           5.0|
|     1| 2024-03-10|Engineering|     4|    Jane|           4.5|
|     2| 2024-01-11|         HR|     4|    Jane|           4.0|
|     2| 2024-03-11|         HR|  NULL|     Sam|           4.0|
|     3| 2024-01-12|      Sales|     3|     Sam|           3.0|
|     4| 2024-02-01|Engineering|     5|    John|           5.0|
+------+-----------+-----------+------+--------+--------------+

